In [ ]:
import sys
import os
import torch
import atexit
import signal

# # 设置 NCCL 环境变量
# os.environ['NCCL_DEBUG'] = 'INFO'
# os.environ['NCCL_IB_DISABLE'] = '1'
# os.environ['NCCL_P2P_DISABLE'] = '1'
# os.environ['NCCL_SHM_DISABLE'] = '1'
# os.environ['NCCL_ALGO'] = 'Ring'  # 改为Ring算法，解决ncclInt8数据类型问题
# os.environ['NCCL_BUFFSIZE'] = '8388608'  # 8MB缓冲区大小

# 添加项目根目录到Python路径
project_root = os.path.dirname(os.getcwd())
# project_root = os.getcwd()
print(f"Adding project root to Python path: {project_root}")
sys.path.insert(0, project_root)

from implement.generator import MTHFVRPGenerator
from implement.environment import MTHFVRPEnv
from implement.evaluation import MTHFVRPEvaluator
from implement.model import TransformerModel
from implement.reinforce_with_batchsample_alg_multigpu import REINFORCELightning as AlgoModel
from implement.trainer import MyCallback, OurTrainer

# 注册退出处理以清理分布式进程组
def cleanup():
    import torch
    if torch. distributed.is_initialized():
        torch.distributed.destroy_process_group()
    torch.cuda.empty_cache()

atexit.register(cleanup)
signal.signal(signal. SIGTERM, lambda s, f: exit(0))
signal.signal(signal. SIGINT, lambda s, f: exit(0))


# === 验证问题生成器 ===
class EvalGenerator():
    def __init__(self, instance):
        self.instance = instance

    def __call__(self, batch_size):
        return self.instance


# 设置使用的GPU设备
os.environ['CUDA_VISIBLE_DEVICES'] = '0'    # 需调用多张GPU时，添加对应设备编号，如 '0,1,2,3'

In [ ]:
# 超参数
'''参考配置：
    batch size 125 * 2卡, 不建议单张卡的batch size小于125,
    node 50, vehicle 10, 或者, node 100 vehicle 15,
    variant: hf_all, no_hf_all, 或者单一variant,
    每epoch训练40个batch, 
    eval size 1000
'''

# PROJECT = "model_v16_n50_v10_bigbatch"  # 项目名称
PROJECT = "test"
VARIANT = "hf_all"      # 项目名称
BATCH_SIZE = 10        # 每次训练使用多少个样本
ACCUMULATE_GRAD_BATCHES = 1  # 梯度累积步数
NODE_NUM = 50          # 节点数量
VEHICLE_NUM = 10         # 车辆数量
TRAIN_BATCH_NUMS = 30   # 每个epoch训练多少个batch
EVAL_SIZE = 1000        # 评估样本数量

# variants = [
#     "cvrp", "vrptw", "vrpbl", "ovrp", "ovrptw",
#     "hfcvrp", "hfvrptw", "hfvrpbl", "hfovrp", "hfovrptw"
# ]

# 1. 创建环境
generator = MTHFVRPGenerator(num_loc=NODE_NUM, variant_preset=VARIANT, vehicle_num=VEHICLE_NUM)   
train_env = MTHFVRPEnv(generator=generator, batch_size=BATCH_SIZE)    # 训练环境

eval_generator = MTHFVRPGenerator(num_loc=NODE_NUM, variant_preset=VARIANT, vehicle_num=VEHICLE_NUM, random_seed=42) 
eval_gen = EvalGenerator(instance=eval_generator(EVAL_SIZE))  # 创建评估问题生成器
eval_env = MTHFVRPEnv(generator=eval_gen, batch_size=EVAL_SIZE)     # 验证环境

In [ ]:
# 2. 加载模型
def load_model(path=None):
    state = train_env.reset()
    input_features = train_env.get_global_features(state)
    state_feature_dims = {}
    for key, feature in input_features["state_feature"].items():  # 直接引用了输入的张量
        feature_dim = feature.shape[-1]
        state_feature_dims[key] = feature_dim
    current_features, _ = train_env.get_current_feature_and_mask(state)
    state_feature_dims['current_feature'] = current_features.shape[-1] - 2  # 减去当前选择节点、车辆的索引维度

    model = TransformerModel(hidden_size=128,
                            n_head=8,
                            encoder_num_layers=6,
                            state_feature_dims=state_feature_dims,
    )

    # 可读取checkpoint
    if path:
        weights = torch.load(path, weights_only=False)
        if 'model_state_dict' in weights:
            model.load_state_dict(weights['model_state_dict']) 
        else:
            model.load_state_dict(weights)    

    return model
    
model = load_model()


# 3. 创建评估器
evaluator = MTHFVRPEvaluator()

In [ ]:
# 4. 创建REINFORCE算法模块
'''
建议参数配置：
    learning_rate=3e-4, 
    warmup_iterations=20, 
    min_lr=2e-4,
    weight_decay=0.01, 
    ent_decay_start=0.4, 
    ent_coef=0.03, 
    clip_cov_range=(0.1, 5.0), 
    cov_grad_detach_prob=0.15, 
    use_num_starts_baseline=True, 
    action_selection_strategy="sampling"
'''
algo = AlgoModel(
    train_env=train_env,
    eval_env=eval_env,
    model=model,
    evaluator=evaluator,
    batch_size=BATCH_SIZE,          # 每次更新使用多少个样本
    train_batch_nums=TRAIN_BATCH_NUMS,   # 一个epoch要训练多少个环境
    learning_rate=3e-4,
    warmup_iterations=20, # Warmup迭代次数
    min_lr=2e-4, # 余弦退火的最小学习率
    weight_decay=0.01,
    ent_decay_start = 0.4,
    ent_coef = 0.03,
    clip_cov_range=(0.1, 5.0),  
    cov_grad_detach_prob=0.15,
    accumulate_grad_batches=ACCUMULATE_GRAD_BATCHES,      # 梯度累积步数
    use_num_starts_baseline=True,    # 是否使用多起点基线
    action_selection_strategy="sampling",  # 动作选择策略: 'sampling' 或 'greedy'
)


# 5. 设置回调
callbacks = [
    # 训练中的验证和保存
    MyCallback(
        eval_env=eval_env,          # 验证环境
        evaluator=evaluator,        # 验证评估器
        val_every_n_epochs=TRAIN_BATCH_NUMS,      # 每隔多少epoch进行一次验证
        save_top_k=1,               # 保存最好的k个模型
        save_dir = "./checkpoints",  # 模型保存目录
        log_dir = "./train_logs",    # 日志保存目录
        filename_prefix=PROJECT,
        monitor_metric="val_mean_reward",
        mode="max",
        verbose=True,             # 是否打印详细信息
        save_last=True,           # 是否保存最后一个模型   
        early_stopping=True,      # 启用早停
        patience=1000,             # 1000次验证没有改善就停止
        min_delta=0.001,          # 最小改善阈值
        use_tensorboard=True,    # 是否使用TensorBoard记录
    ),
]

In [ ]:
# 6. 创建Trainer

trainer = OurTrainer(
    max_epochs=1,       # epoch数量, 建议设置为1000或更大
    callbacks=callbacks,
    accelerator="gpu",
    devices="auto",  # 使用所有可用GPU
    enable_progress_bar=True,
    precision="32", # "16-mixed", "32_all"
    logger=False,  # 关闭默认logger
)

# 7. 开始训练
trainer.fit(algo)